In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_train = pd.read_csv('/kaggle/input/titanic/train.csv').sample(frac=1)
df_test = pd.read_csv('/kaggle/input/titanic/test.csv')

# Preprocessing

In [ ]:
CATEGORICAL_COLUMNS = ['Sex', 'Pclass', 'SibSp', 'Parch', 'Embarked']
NUMERIC_COLUMNS = ['Age', 'Fare', 'PassengerId']

# removing useless cols
useless_cols = ['Name', 'Ticket', 'Cabin']
df_train = df_train.drop(useless_cols, axis=1)
df_test = df_test.drop(useless_cols, axis=1)

# numerical encoding
df_test['Embarked'] = df_test['Embarked'].replace(['S', 'C', 'Q'], [1,2,3])
df_train['Embarked'] = df_train['Embarked'].replace(['S', 'C', 'Q'], [1,2,3])
df_test['Sex'] = df_test['Sex'].replace(['male', 'female'], [-1,1])
df_train['Sex'] = df_train['Sex'].replace(['male', 'female'], [-1,1])

# filling null values with mean
df_train = df_train.fillna(df_train.mean())
df_test = df_test.fillna(df_test.mean())

# normalization of non_cat
df_mean = df_train.loc[:, NUMERIC_COLUMNS].mean()
df_std = df_train.loc[:, NUMERIC_COLUMNS].std()
df_train.loc[:, NUMERIC_COLUMNS] = (df_train.loc[:, NUMERIC_COLUMNS]-df_mean) / df_std
df_test.loc[:, NUMERIC_COLUMNS] = (df_test.loc[:, NUMERIC_COLUMNS]-df_mean) / df_std

# Input
Y = df_train.pop('Survived').to_numpy().reshape(1, -1)
X = df_train.to_numpy().T
X_test = df_test.to_numpy().T

# Forward Propagation

In [ ]:
def forward_propagation(W1, W2, b1, b2, X):
    Z1 = W1 @ X + b1
    A1 = np.tanh(Z1)
    Z2 = W2 @ A1 + b2
    A2 = 1 / (1 + np.exp(-Z2))

    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}
    return cache

# Compute Cost

In [ ]:
def compute_cost(X, Y, A2):
    
    m = X.shape[1]
    J = (-1 / m) * (Y @ np.log(A2).T + (1 - Y) @ np.log(1 - A2).T)
    J = np.squeeze(J)
    
    return J

# Backward Propagation

In [ ]:
def backward_propagation(W1, W2, b1, b2, Y, cache):
    m = Y.shape[1]
    A1 = cache["A1"]
    A2 = cache["A2"]

    dW2 = (1 / m) * (A2 - Y) @ A1.T                     # (1,4) = (1,m).(m,4)
    db2 = (1 / m) * np.sum(A2 - Y, keepdims=True)       # (1,1) = (1,m)

    dZ1 = np.dot(W2.T, A2 - Y) * (1 - np.power(A1, 2))  # (4,3) = (4,1).(1,3)
    dW1 = (1 / m) * dZ1 @ X.T                           # (4,n) = (4,m).(m,n)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)  # (4,1) = sum(4,m)

    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}
    return grads

# Gradient Descent

In [ ]:
def gradieant_descent(W1, W2, b1, b2, X, Y, num_iterations, learning_rate):
    costs = []
    for i in range(num_iterations):
        cache = forward_propagation(W1, W2, b1, b2, X)
        cost = compute_cost(X, Y, cache["A2"])
        grads = backward_propagation(W1, W2, b1, b2, Y, cache)

        W1 = W1 - learning_rate * grads["dW1"]
        b1 = b1 - learning_rate * grads["db1"]
        W2 = W2 - learning_rate * grads["dW2"]
        b2 = b2 - learning_rate * grads["db2"]
        
        param = {"W1": W1,
                 "W2": W2,
                 "b1": b1,
                 "b2": b2}
        
        costs.append(cost)
        print(f'iteration : {i}\t cost : {cost} \n' if i%10 == 0 else "", end = "")

    return param, costs

In [ ]:
# Initialize parameters
n_h = 40
W1 = np.random.randn(n_h, X.shape[0]) * 0.01
b1 = np.zeros((n_h, 1))
W2 = np.random.randn(1, n_h) * 0.01
b2 = np.zeros((1, 1))

# let's train
param, cost = gradieant_descent(W1, W2, b1, b2, X, Y, num_iterations=500, learning_rate=0.1)
# let's Predict
Y_test = forward_propagation(param["W1"], param["W2"], param["b1"], param["b2"], X_test)["A2"]

In [ ]:
plt.plot(cost)

## Submission

In [ ]:
df_submission = pd.DataFrame({'PassengerId': pd.read_csv('/kaggle/input/titanic/test.csv').PassengerId,'Survived': np.squeeze(Y_test >= 0.5).astype(int)})
df_submission.to_csv('my_submission.csv', index=False)